# Decorator Pattern

Attach new behavior to an object dynamically without modifying its original class or affecting other objects of the same class.

    - We have downloading service and we want to add extra functionality or new behaviour to the service like logging and timing. We want to attach this extra functionality at "runtime"

 - Template components of decorator Pattern
    - Component — the common interface (e.g., Beverage with cost()).
    - ConcreteComponent — the core object (Coffee).
    - Decorator — a class that implements Component AND holds a reference to another Component. It forwards requests and adds behavior before/after.
    - ConcreteDecorators — specific decorators (MilkDecorator, SugarDecorator).

In [3]:
from abc import ABC, abstractmethod
import time

## create an interface for the service
class Downloader(ABC):

    @abstractmethod
    def download(self, file_name):
        pass


## concrete downloader
class FileDownloader(Downloader):

    def download(self, file_name):

        print("Starting to download the  file ->", file_name)
        time.sleep(2)
        print("Download finished for file", file_name, "!!!")


### Explosion of subclasses ( NOT RECOMMENDED WAY TO HANDLE)

class FileDownloaderWithLoggingAndTiming(Downloader):
    def download(self, file_name):
        ...

class FileDownloaderWithTiming(Downloader):

    def download(self, file_name):
        start = time.time()
        print("Starting to download the  file ->", file_name)
        time.sleep(2)
        print("Download finished for file", file_name, "!!!")
        end_time = time.time() - start
        print("Time taken: ", end_time - start)

class FileDownloaderWithLogging(Downloader):

    def download(self, file_name):
        print("[LOG] Download started")
        print("Starting to download the  file ->", file_name)
        time.sleep(2)
        print("Download finished for file", file_name, "!!!")
        print("[LOG] Download ended")



In [4]:
filedownloader = FileDownloader()
filedownloader.download("movie.mp4")

Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!


In [26]:
from abc import ABC, abstractmethod
import time

## create an interface for the service
class Downloader(ABC):

    @abstractmethod
    def download(self, file_name):
        pass


## concrete downloader
class FileDownloader(Downloader):

    def download(self, file_name):

        print("Starting to download the  file ->", file_name)
        time.sleep(2)
        print("Download finished for file", file_name, "!!!")


## Interface for a decorator ( actual class and decorator class have the same interface ! )
class DownloaderDecorator(Downloader):

    def __init__(self, downloader : Downloader): # taking the actual object and executing the download on that 
        self._downloader = downloader

    def download(self, file_name):
        self.downloader.download()

## concrete decorator

class LoggingDecorator(DownloaderDecorator):
    ## add the extra functionality

    def download(self, file_name):
        print("[LOG] Download started") # this is decoration
        self._downloader.download(file_name)  # actual execution of the object
        print("[LOG] Download ended") # this is decoration

class TimingDecorator(DownloaderDecorator):
    
    def download(self, file_name):
        start_time = time.perf_counter()
        self._downloader.download(file_name)  # main functionality
        end_time = time.perf_counter() 
        execution_time = end_time - start_time
        print(f"Time taken : {execution_time:.2f} secs ")

 - FileDownloader() and LoggingDecorator(FileDownloader()) are having the same interface therefore they are intercahngeble during runtime as per the requirement

In [14]:
# core functionality without any extra responsibility
downloader = FileDownloader()
downloader.download("movie.mp4")

Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!


In [15]:
# core functionality with any extra responsibility of Logging
downloader = LoggingDecorator(FileDownloader()) # adding a decorator
downloader.download("movie.mp4")

[LOG] Download started
Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!
[LOG] Download ended


In [27]:
# core functionality with any extra responsibility of Timing
downloader = TimingDecorator(FileDownloader()) # adding a decorator
downloader.download("movie.mp4")

Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!
Time taken : 2.01 secs 


In [28]:
# core functionality with any extra responsibility of Logging and Timing
downloader = LoggingDecorator(TimingDecorator(FileDownloader())) # adding a decorator
downloader.download("movie.mp4")

[LOG] Download started
Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!
Time taken : 2.00 secs 
[LOG] Download ended


In [29]:
# core functionality with any extra responsibility of Timing and Logging
downloader = TimingDecorator(LoggingDecorator(FileDownloader())) # adding a decorator
downloader.download("movie.mp4")

[LOG] Download started
Starting to download the  file -> movie.mp4
Download finished for file movie.mp4 !!!
[LOG] Download ended
Time taken : 2.01 secs 


 - Decorator syntax in Python v/s Decorator Pattern


Decorator Syntax we are not flexible to make changes during runtime. We are mofiying the definition of the function/ method

Decorator pattern doenst modify the object itself. But wraps the object in decorators at runtime. We can chose the decorators at runtime dynamically. 

## same example using python decorator syntax

In [30]:
def log_and_time(func):

    def wrapper(*args, **kwargs):

        print("[LOG] Download started")

        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()

        print(f"[TIME] Took {end - start:.2f} seconds")
        print("[LOG] Download finished")

        return result

    return wrapper

@log_and_time
def download(file_name):

    print(f"Downloading {file_name}...")
    time.sleep(2)
    print("Download complete")


# download = log_and_time(download)  # adding the decorator @log_and_time will lead to this



download(file_name="movie.mp4")


# download = log_and_time(download)

Download complete


In [31]:
download = log_and_time(download)  # this line alternate way of representing the decorator syntax

In [32]:
download(file_name="movie.mp4")

[LOG] Download started
Download complete
[TIME] Took 2.01 seconds
[LOG] Download finished


In [33]:
from abc import ABC, abstractmethod

# Component
class Beverage(ABC):
    @abstractmethod
    def cost(self): pass
    @abstractmethod
    def description(self): pass


# ConcreteComponent
class Coffee(Beverage):
    def cost(self): return 100
    def description(self): return "Coffee"


# Decorator base
class BeverageDecorator(Beverage):
    def __init__(self, wrapped: Beverage):
        self.wrapped = wrapped


# Concrete Decorators
class Milk(BeverageDecorator):
    def cost(self): return self.wrapped.cost() + 20
    def description(self): return self.wrapped.description() + ", milk"


class Sugar(BeverageDecorator):
    def cost(self): return self.wrapped.cost() + 10
    def description(self): return self.wrapped.description() + ", sugar"


class Chocolate(BeverageDecorator):
    def cost(self): return self.wrapped.cost() + 30
    def description(self): return self.wrapped.description() + ", chocolate"


# Usage — compose at runtime
order = Chocolate(Milk(Sugar(Coffee())))
print(order.description())  # Coffee, sugar, milk, chocolate
print(order.cost())         # 100 + 10 + 20 + 30 = 160

# How cost() cascades: you call cost() on the OUTERMOST wrapper first, but each
# decorator must ask its inner object before it can add its own price. So the
# call travels INWARD to the core, then accumulates on the way back OUT:
#   Chocolate.cost()
#     -> Milk.cost()
#         -> Sugar.cost()
#             -> Coffee.cost() = 100      (innermost: the real base)
#         -> 100 + 10 = 110               (Sugar adds on the way back)
#     -> 110 + 20 = 130                   (Milk adds)
#   -> 130 + 30 = 160                     (Chocolate adds last)

Coffee, sugar, milk, chocolate
160


# Adaptor Pattern

Convert the interface of a class into another interface clients expect. Adapter lets classes work together that couldn't otherwise because of incompatible interfaces.

- Structure
    - Target — the interface your client code expects (PaymentProcessor).
    - Adaptee — the existing class with a different interface (StripeAPI).
    - Adapter — a class that implements Target and delegates to Adaptee, translating as needed.

In [35]:
# target interface
class PaymentGateway(ABC):

    @abstractmethod
    def process_payment(self, amount):
        pass


## 3rd party APIs that are not compatible with the target interface
class StripeAPI:

    def make_payment(self, amount):
        print(f"Stripe: Paid ₹{amount}")


class PayPalAPI:

    def send_money(self, amount):
        print(f"PayPal: Sent ₹{amount}")


## Create adapter

class StripeAPIAdapter(PaymentGateway):
    def __init__(self, stripe):
        self.stripe = stripe

    def process_payment(self, amount):
        self.stripe.make_payment(amount)

class PayPalAdapter(PaymentGateway):

    def __init__(self, paypal):
        self.paypal = paypal

    def process_payment(self, amount):
        self.paypal.send_money(amount)


In [36]:
gateway = StripeAPIAdapter(StripeAPI())
gateway.process_payment(100)


Stripe: Paid ₹100


In [38]:
gateway = PayPalAdapter(PayPalAPI())
gateway.process_payment(100)

PayPal: Sent ₹100


# Facade Pattern

Provide a unified, simplified interface to a set of interfaces in a subsystem. Facade defines a higher-level interface that makes the subsystem easier to use.

In [ ]:
# Subsystem classes — complex but unchanged
class Projector:
    def turn_on(self): print("Projector ON")
    def set_input(self, src): print(f"Projector input: {src}")

class Screen:
    def lower(self): print("Screen lowered")
    def raise_(self): print("Screen raised")

class Amplifier:
    def turn_on(self): print("Amp ON")
    def set_volume(self, v): print(f"Volume: {v}")
    def set_source(self, src): print(f"Amp source: {src}")

class DVDPlayer:
    def turn_on(self): print("DVD ON")
    def play(self, movie): print(f"Playing '{movie}'")
    def stop(self): print("DVD stopped")

class Lights:
    def dim(self, pct): print(f"Lights at {pct}%")
    def on(self): print("Lights ON")


# Facade
class HomeTheater:
    def __init__(self):
        self.projector = Projector()
        self.screen = Screen()
        self.amp = Amplifier()
        self.dvd = DVDPlayer()
        self.lights = Lights()

    def watch_movie(self, movie):
        self.lights.dim(10)
        self.screen.lower()
        self.projector.turn_on()
        self.projector.set_input("hdmi-2")
        self.amp.turn_on()
        self.amp.set_source("dvd")
        self.amp.set_volume(25)
        self.dvd.turn_on()
        self.dvd.play(movie)

    def end_movie(self):
        self.dvd.stop()
        self.screen.raise_()
        self.lights.on()


# Client code — beautifully simple
theater = HomeTheater()
theater.watch_movie("Interstellar")
# ...
theater.end_movie()

In [42]:
# Subsystem classes — complex but unchanged
class InventoryService:

    def check_stock(self, product):
        print(f"Checking stock for {product}")


class PaymentService:

    def process_payment(self, amount):
        print(f"Payment of ₹{amount} successful")


class ShippingService:

    def create_shipment(self):
        print("Shipment created")


class InvoiceService:

    def generate_invoice(self):
        print("Invoice generated")


class NotificationService:

    def send_confirmation(self):
        print("Confirmation email sent")


## Facade

class CheckoutFacade:

    def __init__(self):
        self.inventory = InventoryService()
        self.payment = PaymentService()
        self.shipping = ShippingService()
        self.invoice = InvoiceService()
        self.notification = NotificationService()

    def place_order(self, product, amount):

        self.inventory.check_stock(product)
        self.payment.process_payment(amount)
        self.shipping.create_shipment()
        self.invoice.generate_invoice()
        self.notification.send_confirmation()

In [43]:
checkout = CheckoutFacade()

checkout.place_order("Laptop", 75000)

Checking stock for Laptop
Payment of ₹75000 successful
Shipment created
Invoice generated
Confirmation email sent


In [44]:

class CPU:

    def start(self):
        print("CPU started")


class Memory:

    def load(self):
        print("Memory loaded")


class HardDisk:

    def read_boot_sector(self):
        print("Boot sector loaded")


class OperatingSystem:

    def start(self):
        print("Operating System started")


class ComputerFacade:

    def __init__(self):
        self.cpu = CPU()
        self.memory = Memory()
        self.disk = HardDisk()
        self.os = OperatingSystem()

    def power_on(self):

        self.cpu.start()
        self.memory.load()
        self.disk.read_boot_sector()
        self.os.start()

computer = ComputerFacade()

computer.power_on()

CPU started
Memory loaded
Boot sector loaded
Operating System started
